# Reproduction notebook: 42_direct_forecaster_absolute_performance_audit

This notebook is retained as an executable provenance record for the anonymous supplementary package. Saved outputs and internal development notes have been removed.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 280)
pd.set_option("display.max_rows", 200)

DATASETS = [
    "Solar",
    "Weather",
    "Electricity",
    "Traffic",
    "Exchange",
    "ETTh1",
]

BACKBONES = [
    "PatchTST",
    "iTransformer",
    "TimeMixer",
    "SegMoE",
]

HORIZONS = [96, 192, 336, 720]

ROOT_CANDIDATES = [
    Path("/data/dataset/strong_forecaster"),
    Path("/data/strong_forecaster"),
]

ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), None)

if ROOT is None:
    raise FileNotFoundError(
        "Could not find strong_forecaster root. Tried:\n"
        + "\n".join(f" - {p}" for p in ROOT_CANDIDATES)
    )

OUT_DIR = ROOT / "direct_forecaster_absolute_performance"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)


In [ ]:
preferred = (
    ROOT
    / "four_backbone_dataset_meta_analysis"
    / "condition_level_selected.csv"
)

if preferred.is_file():
    condition_csv = preferred
else:
    hits = sorted(ROOT.rglob("condition_level_selected.csv"))

    hits = [
        p for p in hits
        if "significance_metadata_recovery" not in str(p)
        and "direct_forecaster_absolute_performance" not in str(p)
    ]

    if not hits:
        raise FileNotFoundError(
            "Could not find condition_level_selected.csv. "
            "Run Experiment 37 v3 first."
        )

    condition_csv = hits[0]

print("Using:", condition_csv)

conditions = pd.read_csv(condition_csv)

required = {
    "Dataset",
    "Backbone",
    "Horizon",
    "MSEGain_pct",
}

missing = required - set(conditions.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {sorted(missing)}"
    )

conditions = conditions[
    conditions["Dataset"].isin(DATASETS)
    & conditions["Backbone"].isin(BACKBONES)
    & conditions["Horizon"].astype(int).isin(HORIZONS)
].copy()

conditions["Horizon"] = conditions["Horizon"].astype(int)

KEY = ["Dataset", "Backbone", "Horizon"]

if len(conditions) != 96:
    raise RuntimeError(
        f"Expected 96 selected conditions; found {len(conditions)}."
    )

if conditions.duplicated(KEY).any():
    raise RuntimeError(
        "Duplicate selected conditions found."
    )

print("Columns available:")
for c in conditions.columns:
    print(" -", c)


In [ ]:
def find_col(df, candidates):
    lookup = {
        str(c).lower().replace("_", "").replace("-", ""): c
        for c in df.columns
    }

    for name in candidates:
        key = (
            str(name)
            .lower()
            .replace("_", "")
            .replace("-", "")
        )

        if key in lookup:
            return lookup[key]

    return None


ALIASES = {
    "DirectMSE": [
        "DirectMSE",
        "Direct_MSE",
        "BaselineMSE",
        "BaseMSE",
        "MSE_Direct",
        "DirectTestMSE",
    ],
    "FinalMSE": [
        "FinalMSE",
        "Final_MSE",
        "ShrinkAdaptiveMSE",
        "AdaptiveMSE",
        "OursMSE",
        "MSE_Final",
        "FinalTestMSE",
    ],
    "DirectMAE": [
        "DirectMAE",
        "Direct_MAE",
        "BaselineMAE",
        "BaseMAE",
        "MAE_Direct",
        "DirectTestMAE",
    ],
    "FinalMAE": [
        "FinalMAE",
        "Final_MAE",
        "ShrinkAdaptiveMAE",
        "AdaptiveMAE",
        "OursMAE",
        "MAE_Final",
        "FinalTestMAE",
    ],
}

detected = {
    canonical: find_col(
        conditions,
        candidates,
    )
    for canonical, candidates in ALIASES.items()
}

print("Detected metric columns:")
for k, v in detected.items():
    print(f" - {k}: {v}")


In [ ]:
absolute = conditions[KEY + ["MSEGain_pct"]].copy()

for canonical, source_col in detected.items():
    if source_col is None:
        absolute[canonical] = np.nan
    else:
        absolute[canonical] = pd.to_numeric(
            conditions[source_col],
            errors="coerce",
        )

absolute["MSESource"] = np.where(
    absolute["DirectMSE"].notna()
    & absolute["FinalMSE"].notna(),
    "Experiment37Selected",
    "Missing",
)

absolute["MAESource"] = np.where(
    absolute["DirectMAE"].notna()
    & absolute["FinalMAE"].notna(),
    "Experiment37Selected",
    "Missing",
)

print(
    "MSE pairs already present:",
    int(
        (
            absolute["DirectMSE"].notna()
            & absolute["FinalMSE"].notna()
        ).sum()
    ),
    "/96",
)

print(
    "MAE pairs already present:",
    int(
        (
            absolute["DirectMAE"].notna()
            & absolute["FinalMAE"].notna()
        ).sum()
    ),
    "/96",
)

display(
    absolute.sort_values(KEY).head(24)
)


In [ ]:
def canon_dataset(x):
    s = str(x).strip().lower().replace("-", "").replace("_", "")

    return {
        "solar": "Solar",
        "solarenergy": "Solar",
        "weather": "Weather",
        "electricity": "Electricity",
        "ecl": "Electricity",
        "traffic": "Traffic",
        "exchange": "Exchange",
        "exchangerate": "Exchange",
        "etth1": "ETTh1",
    }.get(s, str(x).strip())


def canon_backbone(x):
    s = str(x).strip().lower().replace("-", "").replace("_", "")

    return {
        "patchtst": "PatchTST",
        "itransformer": "iTransformer",
        "timemixer": "TimeMixer",
        "segmoe": "SegMoE",
        "segmoeforecast": "SegMoE",
    }.get(s, str(x).strip())


def excluded_path(p):
    s = str(p)

    banned = [
        "four_backbone_dataset_meta_analysis/",
        "significance_metadata_recovery",
        "direct_forecaster_absolute_performance/",
    ]

    return any(token in s for token in banned)


historical_rows = []

for p in ROOT.rglob("*.csv"):
    if excluded_path(p):
        continue

    try:
        if p.stat().st_size > 500 * 1024 * 1024:
            continue
    except Exception:
        continue

    try:
        df = pd.read_csv(p)
    except Exception:
        continue

    c_dataset = find_col(
        df,
        ["Dataset", "Data"],
    )

    c_backbone = find_col(
        df,
        ["Backbone", "Model"],
    )

    c_horizon = find_col(
        df,
        ["Horizon", "PredLen", "pred_len", "H"],
    )

    c_direct_mse = find_col(
        df,
        ALIASES["DirectMSE"],
    )

    c_final_mse = find_col(
        df,
        ALIASES["FinalMSE"],
    )

    if None in {
        c_dataset,
        c_backbone,
        c_horizon,
        c_direct_mse,
        c_final_mse,
    }:
        continue

    c_direct_mae = find_col(
        df,
        ALIASES["DirectMAE"],
    )

    c_final_mae = find_col(
        df,
        ALIASES["FinalMAE"],
    )

    for idx, r in df.iterrows():
        dataset = canon_dataset(
            r[c_dataset]
        )

        backbone = canon_backbone(
            r[c_backbone]
        )

        try:
            horizon = int(
                r[c_horizon]
            )
        except Exception:
            continue

        if dataset not in DATASETS:
            continue

        if backbone not in BACKBONES:
            continue

        if horizon not in HORIZONS:
            continue

        dmse = pd.to_numeric(
            pd.Series([r[c_direct_mse]]),
            errors="coerce",
        ).iloc[0]

        fmse = pd.to_numeric(
            pd.Series([r[c_final_mse]]),
            errors="coerce",
        ).iloc[0]

        if pd.isna(dmse) or pd.isna(fmse):
            continue

        dmae = (
            pd.to_numeric(
                pd.Series([r[c_direct_mae]]),
                errors="coerce",
            ).iloc[0]
            if c_direct_mae is not None
            else np.nan
        )

        fmae = (
            pd.to_numeric(
                pd.Series([r[c_final_mae]]),
                errors="coerce",
            ).iloc[0]
            if c_final_mae is not None
            else np.nan
        )

        historical_rows.append({
            "Dataset": dataset,
            "Backbone": backbone,
            "Horizon": horizon,
            "DirectMSE_recovered": dmse,
            "FinalMSE_recovered": fmse,
            "DirectMAE_recovered": dmae,
            "FinalMAE_recovered": fmae,
            "RecoverySource": str(p),
            "RecoveryRow": int(idx),
        })

historical = pd.DataFrame(
    historical_rows,
    columns=[
        "Dataset",
        "Backbone",
        "Horizon",
        "DirectMSE_recovered",
        "FinalMSE_recovered",
        "DirectMAE_recovered",
        "FinalMAE_recovered",
        "RecoverySource",
        "RecoveryRow",
    ],
)

print(
    "Historical candidate rows:",
    len(historical),
)

if len(historical):
    display(
        historical.sort_values(KEY).head(100)
    )


In [ ]:
gain_lookup = absolute.set_index(KEY)["MSEGain_pct"].to_dict()

resolved_rows = []

if len(historical):
    for key, g in historical.groupby(KEY):
        frozen_gain = gain_lookup.get(
            key,
            np.nan,
        )

        g = g.copy()

        g["RecoveredGain_pct"] = (
            (
                g["DirectMSE_recovered"]
                - g["FinalMSE_recovered"]
            )
            / g["DirectMSE_recovered"]
            * 100.0
        )

        g["GainMismatch"] = (
            g["RecoveredGain_pct"]
            - frozen_gain
        ).abs()

        g["PathLen"] = g["RecoverySource"].map(len)

        g = g.sort_values(
            [
                "GainMismatch",
                "PathLen",
                "RecoverySource",
            ]
        )

        resolved_rows.append(
            g.iloc[0].to_dict()
        )

recovered = pd.DataFrame(
    resolved_rows
)

print(
    "Resolved historical conditions:",
    len(recovered),
)

if len(recovered):
    display(
        recovered.sort_values(KEY).head(96)
    )


In [ ]:
if len(recovered):
    rec_cols = [
        "Dataset",
        "Backbone",
        "Horizon",
        "DirectMSE_recovered",
        "FinalMSE_recovered",
        "DirectMAE_recovered",
        "FinalMAE_recovered",
        "RecoverySource",
        "RecoveredGain_pct",
        "GainMismatch",
    ]

    rec = recovered[rec_cols].copy()
else:
    rec = pd.DataFrame(
        columns=[
            "Dataset",
            "Backbone",
            "Horizon",
            "DirectMSE_recovered",
            "FinalMSE_recovered",
            "DirectMAE_recovered",
            "FinalMAE_recovered",
            "RecoverySource",
            "RecoveredGain_pct",
            "GainMismatch",
        ]
    )

full = absolute.merge(
    rec,
    on=KEY,
    how="left",
    validate="one_to_one",
)

full["DirectMSE_final"] = (
    full["DirectMSE"]
    .combine_first(
        full["DirectMSE_recovered"]
    )
)

full["FinalMSE_final"] = (
    full["FinalMSE"]
    .combine_first(
        full["FinalMSE_recovered"]
    )
)

full["DirectMAE_final"] = (
    full["DirectMAE"]
    .combine_first(
        full["DirectMAE_recovered"]
    )
)

full["FinalMAE_final"] = (
    full["FinalMAE"]
    .combine_first(
        full["FinalMAE_recovered"]
    )
)

full["MSEMetricSource"] = np.where(
    full["DirectMSE"].notna()
    & full["FinalMSE"].notna(),
    "Experiment37Selected",
    np.where(
        full["DirectMSE_final"].notna()
        & full["FinalMSE_final"].notna(),
        "HistoricalRecovery",
        "Missing",
    ),
)

full["MAEMetricSource"] = np.where(
    full["DirectMAE"].notna()
    & full["FinalMAE"].notna(),
    "Experiment37Selected",
    np.where(
        full["DirectMAE_final"].notna()
        & full["FinalMAE_final"].notna(),
        "HistoricalRecovery",
        "Missing",
    ),
)

full["AbsoluteMSEImprovement"] = (
    full["DirectMSE_final"]
    - full["FinalMSE_final"]
)

full["RecomputedMSEGain_pct"] = (
    full["AbsoluteMSEImprovement"]
    / full["DirectMSE_final"]
    * 100.0
)

full["GainConsistencyError"] = (
    full["RecomputedMSEGain_pct"]
    - full["MSEGain_pct"]
).abs()

display(
    full.sort_values(KEY)
)

full.to_csv(
    OUT_DIR / "direct_final_absolute_metrics_96_conditions.csv",
    index=False,
)


In [ ]:
MSE_TOL_PCT = 0.01

mse_complete = (
    full["DirectMSE_final"].notna()
    & full["FinalMSE_final"].notna()
)

mae_complete = (
    full["DirectMAE_final"].notna()
    & full["FinalMAE_final"].notna()
)

bad_gain = full[
    mse_complete
    & (
        full["GainConsistencyError"]
        > MSE_TOL_PCT
    )
].copy()

print(
    "Complete MSE pairs:",
    int(mse_complete.sum()),
    "/96",
)

print(
    "Complete MAE pairs:",
    int(mae_complete.sum()),
    "/96",
)

print(
    "MSE gain consistency failures:",
    len(bad_gain),
)

if len(bad_gain):
    display(
        bad_gain[
            KEY
            + [
                "MSEGain_pct",
                "DirectMSE_final",
                "FinalMSE_final",
                "RecomputedMSEGain_pct",
                "GainConsistencyError",
                "MSEMetricSource",
                "RecoverySource",
            ]
        ]
    )

    raise RuntimeError(
        "Absolute MSE recovery contains gain-inconsistent rows."
    )

print("PASS: all recovered MSE pairs are consistent with frozen gains.")


In [ ]:
summary_rows = []

for dataset in DATASETS:
    for backbone in BACKBONES:
        g = full[
            (full["Dataset"] == dataset)
            & (full["Backbone"] == backbone)
        ].sort_values("Horizon")

        row = {
            "Dataset": dataset,
            "Backbone": backbone,
            "MeanDirectMSE_overH": (
                float(g["DirectMSE_final"].mean())
                if g["DirectMSE_final"].notna().any()
                else np.nan
            ),
            "MeanFinalMSE_overH": (
                float(g["FinalMSE_final"].mean())
                if g["FinalMSE_final"].notna().any()
                else np.nan
            ),
            "MeanMSEGain_pct_overH": float(
                g["MSEGain_pct"].mean()
            ),
            "Wins": int(
                (g["MSEGain_pct"] > 0).sum()
            ),
            "Ties": int(
                (g["MSEGain_pct"].abs() <= 1e-12).sum()
            ),
            "Losses": int(
                (g["MSEGain_pct"] < 0).sum()
            ),
        }

        if g["DirectMAE_final"].notna().all():
            row["MeanDirectMAE_overH"] = float(
                g["DirectMAE_final"].mean()
            )
            row["MeanFinalMAE_overH"] = float(
                g["FinalMAE_final"].mean()
            )
        else:
            row["MeanDirectMAE_overH"] = np.nan
            row["MeanFinalMAE_overH"] = np.nan

        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)

display(summary)

summary.to_csv(
    OUT_DIR / "dataset_backbone_absolute_summary.csv",
    index=False,
)


In [ ]:
paper = full[
    [
        "Dataset",
        "Backbone",
        "Horizon",
        "DirectMSE_final",
        "FinalMSE_final",
        "MSEGain_pct",
        "DirectMAE_final",
        "FinalMAE_final",
        "MSEMetricSource",
        "MAEMetricSource",
    ]
].copy()

paper = paper.sort_values(
    ["Dataset", "Backbone", "Horizon"]
)

display(paper)

paper.to_csv(
    OUT_DIR / "paper_absolute_metrics_96_conditions.csv",
    index=False,
)

latex = paper.copy()

for c in [
    "DirectMSE_final",
    "FinalMSE_final",
    "DirectMAE_final",
    "FinalMAE_final",
]:
    latex[c] = latex[c].map(
        lambda v: (
            ""
            if pd.isna(v)
            else f"{v:.4f}"
        )
    )

latex["MSEGain_pct"] = latex["MSEGain_pct"].map(
    lambda v: f"{v:+.3f}"
)

(OUT_DIR / "paper_absolute_metrics_96_conditions.tex").write_text(
    latex.to_latex(
        index=False,
        escape=False,
    ),
    encoding="utf-8",
)

print("Saved final CSV and LaTeX table.")


In [ ]:
print("=" * 118)
print("EXPERIMENT 42 — DIRECT FORECASTER ABSOLUTE-PERFORMANCE AUDIT")
print("=" * 118)

print(
    f"Complete MSE pairs: "
    f"{int(mse_complete.sum())}/96"
)

print(
    f"Complete MAE pairs: "
    f"{int(mae_complete.sum())}/96"
)

print(
    "Maximum MSE-gain consistency error:",
    (
        float(
            full.loc[
                mse_complete,
                "GainConsistencyError",
            ].max()
        )
        if mse_complete.any()
        else np.nan
    ),
)

print("\nMSE source breakdown:")
print(
    full[
        "MSEMetricSource"
    ].value_counts(
        dropna=False
    ).to_string()
)

print("\nMAE source breakdown:")
print(
    full[
        "MAEMetricSource"
    ].value_counts(
        dropna=False
    ).to_string()
)

missing_mse = full[
    ~mse_complete
]

missing_mae = full[
    ~mae_complete
]

if len(missing_mse):
    print("\nMissing MSE conditions:")
    print(
        missing_mse[
            KEY + ["MSEGain_pct"]
        ]
        .sort_values(KEY)
        .to_string(index=False)
    )

if len(missing_mae):
    print(
        f"\nMAE missing for "
        f"{len(missing_mae)}/96 conditions."
    )

print("\nOutputs:")
for name in [
    "direct_final_absolute_metrics_96_conditions.csv",
    "dataset_backbone_absolute_summary.csv",
    "paper_absolute_metrics_96_conditions.csv",
    "paper_absolute_metrics_96_conditions.tex",
]:
    print(" -", OUT_DIR / name)
